IMPORTING LIBRARIES

In [2]:
import pandas as pd
import  numpy as np
import re
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords", download_dir="/tmp/nltk_data")  
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_selection import SelectFromModel

[nltk_data] Downloading package stopwords to /tmp/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


LOADING DATASET

In [3]:
train_data = pd.read_csv("C:/Users/User/Desktop/Genre Classification Dataset/train_data.txt", sep=":::", engine="python", names=["ID","Title","Genre","Description"])
test_data = pd.read_csv("C:/Users/User/Desktop/Genre Classification Dataset/test_data_solution.txt", sep=":::", engine="python", names=["ID","Title","Description"])
train_data.head()


,ID,Title,Genre,Description
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his do...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous r...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fie...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends mee...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-re...


CHECKING FOR MISSIMG DATA

In [4]:
train_data.isnull().sum()
train_data.duplicated().sum()

np.int64(0)

MAKING GENRES OF SAME CHARACTER

In [5]:
train_data["Genre"] = train_data["Genre"].str.strip().str.lower()

CLEANING DATASET

In [6]:
stop_words = set(stopwords.words("english"))

def cleaned_set(text):
    if isinstance(text,str):# Ensure text is valid
        text = text.lower()# Convert to lowercase
        text = re.sub(r"[^\w\s]","",text)# Remove punctuation but keep words and spaces
        text = re.sub(r"\d+","",text)# Remove numbers
        text = re.sub(r"\s+"," ",text).strip()# Remove extra spaces
        words = text.split()
        words = [word for word in words if word not in stop_words]# Remove stopwords
        return " ".join(words)
    return ""  # Return empty string if text is missing

train_data["new_Description"]= train_data["Description"].apply(cleaned_set)
test_data["new_Description"] = test_data["Description"].apply(cleaned_set)

In [7]:
#print(train_data)

In [8]:
#print(test_data.shape)
#print(train_data.shape)
#print(test_data) 

FEATURE SCALING

In [9]:
ftf = TfidfVectorizer(
    max_features = 20000,
    stop_words = 'english',
    ngram_range = (1,3),
    sublinear_tf = True,
    min_df = 5,
    max_df = 0.95
)

x_train = ftf.fit_transform(train_data["new_Description"])
x_test = ftf.transform(test_data["new_Description"])


In [10]:
#print(test_data["new_Description"].isnull().sum())
#print(x_train.shape,x_test.shape)
#print("Sample features: ", ftf.get_feature_names_out()[:20])
#print(x_train)

In [11]:
#print(x_test)

ENCODING GENTRE LABELS

In [12]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(train_data["Genre"])


In [13]:
print(y_train[:10])
genre_position = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
print(genre_position)

[ 8 24  1  8  8  7  5  6 18 13]
{'action': np.int64(0), 'adult': np.int64(1), 'adventure': np.int64(2), 'animation': np.int64(3), 'biography': np.int64(4), 'comedy': np.int64(5), 'crime': np.int64(6), 'documentary': np.int64(7), 'drama': np.int64(8), 'family': np.int64(9), 'fantasy': np.int64(10), 'game-show': np.int64(11), 'history': np.int64(12), 'horror': np.int64(13), 'music': np.int64(14), 'musical': np.int64(15), 'mystery': np.int64(16), 'news': np.int64(17), 'reality-tv': np.int64(18), 'romance': np.int64(19), 'sci-fi': np.int64(20), 'short': np.int64(21), 'sport': np.int64(22), 'talk-show': np.int64(23), 'thriller': np.int64(24), 'war': np.int64(25), 'western': np.int64(26)}


SPLITTING TRAIN DATASET

In [14]:
X_train_split,X_val,y_train_split,y_val = train_test_split(x_train, y_train, train_size=0.8, test_size=0.2,random_state = 42)

In [15]:
print("Training set shape: ", X_train_split.shape,y_train_split.shape)
print("Validatiion set shape: ", X_val.shape,y_val.shape)
print("Test set shape (Given to us): ", x_test.shape)

Training set shape:  (43371, 20000) (43371,)
Validatiion set shape:  (10843, 20000) (10843,)
Test set shape (Given to us):  (54200, 20000)


LOGISTIC REGRESSION MODEL

In [ ]:
model = LogisticRegression(max_iter=10000,random_state=42,C=3.0)
model.fit(X_train_split,y_train_split)
y_prediction = model.predict(X_val)
accuracy= accuracy_score(y_val,y_prediction)
print(f"Validation Accuracy: {accuracy:.4f}")

In [17]:
#print("Classification Report :\n", classification_report(y_val,y_prediction))